> Experiments with [model.signal](../trapallada/model/signal.py)  module

In [ ]:
import pathlib

import yaml

import jax.random as jr

import trapallada.data
import trapallada.model.nn
import trapallada.util
import trapallada.core

In [ ]:
%load_ext autoreload
%autoreload 2

Directory containing the data

In [ ]:
# input_directory = pathlib.Path.cwd().parent / 'samples' / 'data' / 'mini_rate'
input_directory = pathlib.Path.cwd().parent / 'data' / 'rate_backv'
assert input_directory.exists()
input_directory

Data is stored in two separate files:

- data itself, in the form of an `Array`

- metadata in a *Pandas* `DataFrame`

In [ ]:
signals_file = input_directory / 'signals.pt'
assert signals_file.exists()

masks_file = input_directory / 'masks.pt'
assert masks_file.exists()

metadata_file = input_directory / 'metadata.parquet'
assert metadata_file.exists

It it turned into a *PyTorch* `Dataset`

In [ ]:
ds = trapallada.data.SignalsDataset(signals_file, metadata_file, masks_file, normalize=True)
len(ds), ds[0][1].shape

A parameters file that specifies an experiment

In [ ]:
parameters_file = pathlib.Path.cwd().parent / 'parameters.yaml'
# parameters_file = pathlib.Path.cwd().parent / 'samples' / 'fft_vanilla_mini.yaml'
assert parameters_file.exists()

It is read into a `dic`

In [ ]:
with parameters_file.open() as f:
    parameters = yaml.safe_load(f)

In [ ]:
%pycat {parameters_file}

`DataLoaders` are derived from the `Dataset`

In [ ]:
train_dl, validation_dl = trapallada.data.dataloaders_from_dataset(
    ds,
    parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    parameters['training']['batch_size'], parameters['PRNG seeds']['training dataloader'])


How many samples per batch?

In [ ]:
batch = next(iter(train_dl))
len(batch)

The *index component* in the first batch

In [ ]:
batch[0]

The *x component* in the 1st element of the `Dataset`

In [ ]:
signals_len = ds[0][1].shape
signals_len

What are the dimensions of every input?

In [ ]:
tuple(ds[0][1].shape)

Both *encoder* and *decoder* also require the *input size* as a parameter: it's inferred from the `Dataset` and included in the corresponding `dict`s

In [ ]:
input_size = list(ds[0][1].shape)
for m in ['encoder', 'decoder']:
        parameters['VAE'][m]['parameters']['input_size'] = input_size

In [ ]:
x = batch[1][0].numpy()
encoder = trapallada.model.nn.Encoder(
    **parameters['VAE']['encoder']['parameters'],
    key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['encoder']))
encoder.debug_call(x)
print(encoder)

In [ ]:
x.shape

In [ ]:
import equinox as eqx
eqx.nn.Conv1d(in_channels=x.shape[0], out_channels=1, kernel_size=3, stride=1, padding=1, key=jr.PRNGKey(42))(x).shape

In [ ]:
encoder.layers_spatial_sizes

In [ ]:
z = encoder(x)
z.shape

In [ ]:
z_mean = jr.normal(jr.PRNGKey(42), (parameters['model']['dim_z'],))
decoder = trapallada.model.nn.Decoder(
    **(parameters['VAE']['decoder']['parameters'] | {'layers_spatial_sizes': encoder.layers_spatial_sizes}),
    key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['decoder']))
decoder.debug_call(z_mean)

In [ ]:
vae = trapallada.model.nn.VAE(encoder, decoder)
x_pred, z_mean, z_sd = vae(x, jr.PRNGKey(42))
x_pred.shape, z_mean.shape, z_sd.shape

In [ ]:
model, results_df = trapallada.core.fit_vae_on_signals(train_dl, validation_dl, parameters)

# Conditional VAE

In [ ]:
cond_decoder = trapallada.model.nn.ConditionalDecoder(
    **(parameters['VAE']['decoder']['parameters'] | {'input_size': [1, input_size[1]]}| {'layers_spatial_sizes': encoder.layers_spatial_sizes}),
    key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['decoder']))
# cond_decoder = trapallada.model.nn.ConditionalDecoder(decoder, key=jr.PRNGKey(parameters['PRNG seeds']['model initialization']['decoder']))
# cond_decoder.z_decoder.layers[0].weight.shape
# cond_decoder.z_decoder.layers[-1].out_channels

In [ ]:
cond_decoder(z_mean, x[-1:])
# x.shape, z.shape

In [ ]:
cvae = trapallada.model.nn.CVAE(encoder, cond_decoder)
x_pred, z_mean, z_sd = cvae(x, jr.PRNGKey(42))
x_pred.shape, z_mean.shape, z_sd.shape